# Aircast Phase 2 — Deep Learning and Calibrated Hybrid PM2.5 Forecasting

Run every cell in order in Google Colab or your local Jupyter environment.

### Overview of Phase 2 Deep Learning Architecture:
* **Recurrent Sequence Models**: Evaluates **LSTM** and **Bi-LSTM** architectures for temporal time-series representation.
* **Temporal Attention**: Additive attention mechanism across past sliding sequence windows ($L=24$ hours) to identify critical weather transitions.
* **Delta vs. Level Formulation**: Models target level $y_{t+1}$ directly or learns a residual delta correction $\Delta y = y_{t+1} - y_t$ against the latest reading.
* **Probabilistic Quantile Regression**: Computes 80% uncertainty prediction intervals $[q_{10}, q_{90}]$ using Pinball Loss to quantify confidence during volatile pollution regimes.
* **Calibrated Hybrid Forecaster**: Pairs a frozen classical base model (XGBoost) with a deep neural residual correction block.
* **Rigorous Validation Protocol**: Hyperparameter selection on 2013 validation folds, paired attention/directionality ablations across 3 predefined random seeds (42, 137, 2026), and backtests on 2014 holdout.

In [ ]:
import base64, io, json, os, sys, zipfile
from hashlib import sha256
from pathlib import Path
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if (Path.cwd() / 'src/air_prediction').is_dir():
    ROOT = Path.cwd()
elif (Path.cwd().parent / 'src/air_prediction').is_dir():
    ROOT = Path.cwd().parent
else:
    ROOT = Path('/content/air_prediction') if IN_COLAB else Path.cwd() / 'air_prediction'

SOURCE_BUNDLE = '...'
SOURCE_BUNDLE_SHA256 = '38247cf8ebd5bb629df359686be1a68f1e0a8119b64a4b26ba15747a02c43e03'
SOURCE_FILES = {'pyproject.toml': '3d00ec5dd0c0f08fe91ed282d9a2da641c818f72b94f5f2cd424d0146e1d7868', 'requirements.txt': 'f6522a3d253d9cee60ca6422c69e7b4c335ee975b2e677a515211820a2dbbe53', 'requirements-phase1.txt': '2a14a8633b1b3bfe05d74b7c920bcb9e8af5ae7dd7526743812c592841ab0557', 'requirements-phase2.txt': 'f080052d966f9b3b339a554c3a885313c0878ac049bc246abdb3c1599680b3df', 'LICENSE': 'e2cc2416866c2512dddb8cf4052dbf1b95610a028d9aae1cd6936c96491a3528', 'configs/phase1.toml': '50d16432bf840dea6b4be31de173d74e7d9a4ade34b0a2ef6ba0f11ab29df10a', 'configs/phase2_research.toml': 'd212393849f43ac11379aabd3bfbe6a074f2dcb0deb04390805b66b0aef68b35', 'configs/demo.toml': '7d2a1f7dbc94f9b69e10e918747b557135242cc98f1a3322bf0d5999c7bf1be9', 'data/README.md': '3343f117cce21bc207151f563ac51f47d02876076a33c5307d9290dbee89d132', 'docs/methodology.md': '8a0b85aa16667fe864f5402e65516f4091d462a19dad1208edaeaa92d748bde5', 'docs/phase2_research.md': 'a34ca8eed2fd94ffb478bc729a58cba1e7b51a98e73356c3da983a58a67a5b2d', 'tests/conftest.py': 'fb423199ad33600b89bb4912a1d67164bcad9ce9b70b2195cc06de171ba4a73f', 'tests/test_forecasting.py': '7ea79d7051614b9bc9c646e68e95981086589a37159cc78d5f5915605bd36ec8', 'tests/test_data_and_integration.py': '1b648af07a64a2cd24729559a0a7e394d8728bc4c14613144b486c91966491b1', 'tests/test_phase2.py': '31545f87f1e1e6dd646bb0f092a2d5b49f6d007af9979bd351d6968dc456ef03', 'tests/test_phase2_research.py': '134a9eab8406cf90ca6ca9cb95bb00311e2516ee140610b9f642134a931ae6bb', 'src/air_prediction/__init__.py': 'd8814eeb3590fdcb355af8851579fbf238a9bf0fed2167aed03c54dc696f0059', 'src/air_prediction/__main__.py': '6d8b7d7846a845059d7a3107143f11131f63c5511d669b44085b15ec5e3d2279', 'src/air_prediction/cli.py': '1d03ea264025673d4b702816b3895b4bfa73c40f842c92ab58ab4cad6e111ef1', 'src/air_prediction/config.py': 'b30a1ff122439f287167e5d5c76a098d77912d5d2b8909e648954ac7f76b3d69', 'src/air_prediction/data.py': '024f8c62c7cc7b01529f7efcc9b0a8cba4058f2658261c267db364a95f087ba8', 'src/air_prediction/deep_learning.py': '35c85d54db34064e48adbe2a58f1fcd747082a77ac4ea6c03c1631af373edf9f', 'src/air_prediction/demo.py': '711dca94046382f98902f5506091e58a719ae31bd4b11ef898a0c907a420b824', 'src/air_prediction/eda.py': '283955e44e25e21c76d434ddf34c9078cdd6ee37c840fa9f44200afc20f93043', 'src/air_prediction/experiment.py': '44170084233e8592a8a3c2ff8a24d8506240c6b671ea0c1567af6d6a712a05d2', 'src/air_prediction/features.py': 'ea9277347a6a4508f81c446267663dbdb3577cfcacc19112a9cac50fbc141831', 'src/air_prediction/metrics.py': 'c05874f36b670169b08ad266c95a3b7b022756f9d74cd43834d235beae111bdb', 'src/air_prediction/models.py': '3b13704e78234cbd55ef69f012ecd589ceb109ffc5973a49a27c4817d9c7c300', 'src/air_prediction/phase2.py': 'ca4722ff5bba885f89b98f7f371b4c22e215dd41d05e09ede4fa979ca6f8ff4f', 'src/air_prediction/phase2_research.py': 'd697510d6f52d48aa90cbff8cda77d903d6c472a4873653cda61c1e02587f98b', 'src/air_prediction/predict.py': '0e45a56f932e2bc2331ad4c24eea8d0c24dfbc0f25af590637faf1ad0dfd7bcb', 'src/air_prediction/results.py': '44543efe599b1f4853ea61f9023c31b7a74b0d48b579576863685febf8eadc19', 'src/air_prediction/splits.py': '91e59bf69d2616d9f46dbb13da99ef4765da07d52591985bdeb6482ed08680b1'}
snapshot = base64.b64decode(SOURCE_BUNDLE)
assert sha256(snapshot).hexdigest() == SOURCE_BUNDLE_SHA256
ROOT.mkdir(parents=True, exist_ok=True)
if not (ROOT / 'src/air_prediction').is_dir():
    with zipfile.ZipFile(io.BytesIO(snapshot)) as archive:
        for name in SOURCE_FILES.keys():
            if name in archive.namelist():
                content = archive.read(name)
                target = (ROOT / name).resolve()
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(content)
for name, expected in SOURCE_FILES.items():
    target = ROOT / name
    if target.exists() and name.startswith(('src/', 'configs/')):
        assert sha256(target.read_bytes()).hexdigest() == expected, 'Source differs: ' + name
os.chdir(ROOT)
for key in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS', 'VECLIB_MAXIMUM_THREADS'):
    os.environ.setdefault(key, '1')
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / '.cache/matplotlib'))
sys.path.insert(0, str(ROOT / 'src'))
print('Aircast Phase 2 environment initialized.')
print('Source snapshot verified:', SOURCE_BUNDLE_SHA256[:16] + '...')
print('Runtime:', sys.version.split()[0], '| Colab:', IN_COLAB)


## 1. Environment & Deep Learning Dependencies

Installs PyTorch, XGBoost, scikit-learn, pandas, numpy, and matplotlib. Detects hardware acceleration (GPU/MPS/CPU).

In [ ]:
import subprocess
INSTALL_DEPENDENCIES = IN_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-phase2.txt'], check=True)
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
try:
    from IPython.display import display, Image as NotebookImage
except ImportError:
    def display(val): print(val)
    NotebookImage = None

device = torch.device('cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu'))
print(f'PyTorch Version : {torch.__version__}')
print(f'Computation Dev : {device}')
print(f'CUDA Available  : {torch.cuda.is_available()}')


## 2. Dataset Loading & Integrity Audit

Loads the historical Beijing hourly PM2.5 dataset (UCI Machine Learning Repository). If running in a fresh Colab environment, it automatically downloads and verifies the exact 43,824-hour archive against its cryptographic SHA-256 signature.

In [ ]:
from air_prediction.config import load_config
from air_prediction.data import FILENAME, download_dataset, load_dataset
from air_prediction.phase2_research import load_research_settings

CONFIG = load_config(ROOT / 'configs/phase1.toml')
SETTINGS = load_research_settings(ROOT / 'configs/phase2_research.toml')
DATA = ROOT / 'data/raw' / FILENAME
if not DATA.exists():
    download_dataset(DATA)
dataset = load_dataset(DATA)

print('Dataset Audit:')
print(f"  Rows total       : {dataset.audit['hourly_rows']:,}")
print(f"  Missing PM2.5    : {dataset.audit['missing_by_column']['pm2.5']} ({dataset.audit['missing_by_column']['pm2.5']/dataset.audit['hourly_rows']*100:.2f}%)")
print(f"  Date range       : {dataset.audit['start']} to {dataset.audit['end']}")
print(f"  SHA-256 checksum : {dataset.audit['sha256']}")


## 3. Sliding Sequence Windows for Deep Learning

Standard feedforward models use single tabular feature vectors. In recurrent deep learning, we map the multivariate hourly series into 3D tensors:
- Input sequence $X \in \mathbb{R}^{B \times L \times D}$, where $B$ is batch size, $L=24$ hours of historical context, and $D$ is feature dimensionality (meteorology + lag indicators + cyclics).
- Target $y \in \mathbb{R}^{B \times 1}$, representing the concentration at origin $t + 1$.

In [ ]:
from air_prediction.deep_learning import DLConfig, DeepLearningPipeline, SequenceAirDataset, TemporalForecaster
from air_prediction.features import build_features

features_df = build_features(dataset.frame)
dl_pipeline = DeepLearningPipeline(DLConfig(seq_len=24, hidden_dim=32, num_layers=1, architecture='bilstm'))
train_mask = (features_df.index >= pd.Timestamp(CONFIG.train_start)) & (features_df.index < pd.Timestamp(CONFIG.validation_start))
dl_pipeline.fit(features_df, features_df['target_1h'], train_mask=train_mask)

print(f"Fitted feature columns : {len(dl_pipeline.columns_)} features")
print(f"Numeric features       : {len(dl_pipeline.numeric_columns_)}")
print(f"Categorical features   : {list(dl_pipeline.category_levels_.keys())}")


## 4. Model Architecture & Attention Mechanism

We construct `TemporalForecaster`, which supports LSTM, GRU, and Bidirectional LSTM backbones.
When `use_attention=True`, an additive attention layer computes a dynamic weighting across all $L=24$ sequence time steps:
$$\alpha_t = \frac{\exp(e_t)}{\sum_{k=1}^L \exp(e_k)}, \quad e_t = v^T \tanh(W h_t)$$

In [ ]:
input_dim = len(dl_pipeline.columns_)
lstm_model = TemporalForecaster(input_dim=input_dim, hidden_dim=32, num_layers=1, architecture='lstm', use_attention=True)
bilstm_model = TemporalForecaster(input_dim=input_dim, hidden_dim=32, num_layers=1, architecture='bilstm', use_attention=False)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Architecture 1: LSTM with Temporal Attention")
print(lstm_model)
print(f"Total Trainable Parameters: {count_parameters(lstm_model):,}\n")

print("Architecture 2: Bi-LSTM (Bidirectional)")
print(bilstm_model)
print(f"Total Trainable Parameters: {count_parameters(bilstm_model):,}")


## 5. Live Deep Learning Training & Epoch Loss Curves

Here we train a representative neural sequence model using the PyTorch Adam optimizer, MSE loss, and delta target mode. We log train and validation loss at each epoch to generate learning curves.

In [ ]:
from torch.utils.data import DataLoader

# Prepare train and validation subsets
val_mask = (features_df.index >= pd.Timestamp(CONFIG.validation_start)) & (features_df.index < pd.Timestamp(CONFIG.holdout_start))
X_train_mat = dl_pipeline._prepare_matrix(features_df[train_mask])
y_train = features_df.loc[train_mask, 'target_1h'].to_numpy(dtype=np.float32)

X_val_mat = dl_pipeline._prepare_matrix(features_df[val_mask])
y_val = features_df.loc[val_mask, 'target_1h'].to_numpy(dtype=np.float32)

train_dataset = SequenceAirDataset(X_train_mat, y_train, seq_len=24)
val_dataset = SequenceAirDataset(X_val_mat, y_val, seq_len=24)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)

train_forecaster = TemporalForecaster(input_dim=X_train_mat.shape[1], hidden_dim=32, architecture='bilstm', use_attention=True)
optimizer = torch.optim.Adam(train_forecaster.parameters(), lr=0.001)
criterion = nn.MSELoss()

epochs = 6
train_losses, val_losses = [], []
print(f"Training Bi-LSTM with Temporal Attention for {epochs} epochs...")

for epoch in range(1, epochs + 1):
    train_forecaster.train()
    total_loss, total_count = 0.0, 0
    for X_batch, y_batch in train_loader:
        mask = torch.isfinite(y_batch)
        if not mask.any():
            continue
        optimizer.zero_grad()
        pred, _ = train_forecaster(X_batch)
        loss = criterion(pred[mask], y_batch[mask])
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * mask.sum().item()
        total_count += mask.sum().item()

    train_rmse = np.sqrt(total_loss / max(1, total_count))
    train_losses.append(train_rmse)

    # Validation phase
    train_forecaster.eval()
    val_loss, val_count = 0.0, 0
    with torch.no_grad():
        for X_b, y_b in val_loader:
            vmask = torch.isfinite(y_b)
            if not vmask.any():
                continue
            p, _ = train_forecaster(X_b)
            vloss = criterion(p[vmask], y_b[vmask])
            val_loss += vloss.item() * vmask.sum().item()
            val_count += vmask.sum().item()
    val_rmse = np.sqrt(val_loss / max(1, val_count))
    val_losses.append(val_rmse)
    print(f"Epoch {epoch}/{epochs} - Train RMSE: {train_rmse:.3f} µg/m³ | Val RMSE: {val_rmse:.3f} µg/m³")

# Plot Loss Curves
plt.figure(figsize=(8, 4))
plt.plot(range(1, epochs + 1), train_losses, 'o-', label='Training Loss (RMSE)', color='#2563eb')
plt.plot(range(1, epochs + 1), val_losses, 's--', label='Validation Loss (RMSE)', color='#dc2626')
plt.title('Deep Learning Training Progression: Bi-LSTM with Attention', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('RMSE (µg/m³)')
plt.grid(True, linestyle=':', alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()


## 6. Attention Weights & Model Interpretability

Deep learning models are often viewed as black boxes. By examining the attention weights $\alpha_t$, we can see which past hours the model focused on when issuing its forecast.

In [ ]:
train_forecaster.eval()
sample_seq = X_val_mat[:24]
sample_tensor = torch.tensor(sample_seq, dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    _, attn_weights = train_forecaster(sample_tensor)

if attn_weights is not None:
    weights = attn_weights.squeeze(0).numpy()
    hours_back = [f"t-{23-i}h" for i in range(24)]

    plt.figure(figsize=(10, 3.5))
    bars = plt.bar(hours_back, weights, color='#3b82f6', edgecolor='#1d4ed8', alpha=0.85)
    plt.title('Learned Temporal Attention Across 24-Hour Sequence Window', fontsize=12, fontweight='bold')
    plt.xlabel('Historical Observation Horizon')
    plt.ylabel('Normalized Attention Weight')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()
    print(f"Peak Attention Horizon: {hours_back[int(np.argmax(weights))]} (Weight: {weights.max():.4f})")


## 7. Probabilistic Forecasting: Quantile Regression & Prediction Intervals

Standard point forecasts ($\hat{y}$) do not convey epistemic or aleatoric uncertainty. In operational air quality management, decision-makers need **prediction intervals** (e.g., an 80% confidence bound).

We train a multi-quantile model using **Pinball Loss**:
$$\mathcal{L}_q(y, \hat{y}_q) = \max\Big(q(y - \hat{y}_q), (1 - q)(\hat{y}_q - y)\Big)$$

By jointly outputting the **10th ($q_{10}$)**, **50th ($q_{50}$, median)**, and **90th ($q_{90}$)** percentiles, the network provides an **80% prediction interval** $[q_{10}, q_{90}]$.

In [ ]:
from air_prediction.deep_learning import PinballLoss, empirical_coverage

q_forecaster = TemporalForecaster(input_dim=X_train_mat.shape[1], hidden_dim=32, architecture='bilstm', use_attention=True, output_dim=3)
q_optimizer = torch.optim.Adam(q_forecaster.parameters(), lr=0.001)
q_criterion = PinballLoss(quantiles=(0.1, 0.5, 0.9))

print("Training Multi-Quantile Forecaster (q=0.10, 0.50, 0.90)...")
q_epochs = 5
for epoch in range(1, q_epochs + 1):
    q_forecaster.train()
    total_q_loss, count = 0.0, 0
    for X_b, y_b in train_loader:
        mask = torch.isfinite(y_b).squeeze(-1)
        if not mask.any():
            continue
        q_optimizer.zero_grad()
        preds, _ = q_forecaster(X_b)
        loss = q_criterion(preds[mask], y_b[mask])
        loss.backward()
        q_optimizer.step()
        total_q_loss += loss.item() * mask.sum().item()
        count += mask.sum().item()
    print(f"Epoch {epoch}/{q_epochs} - Pinball Loss: {total_q_loss / max(1, count):.4f}")

# Evaluate on an example validation segment (168 hours = 1 week)
q_forecaster.eval()
test_slice = 168
all_q_preds = []
with torch.no_grad():
    for i in range(24, test_slice):
        sub_x = torch.tensor(X_val_mat[i-24:i], dtype=torch.float32).unsqueeze(0)
        p, _ = q_forecaster(sub_x)
        all_q_preds.append(p.squeeze(0).numpy())

q_matrix = np.array(all_q_preds)
# Ensure monotonicity q10 <= q50 <= q90
q_matrix = np.sort(q_matrix, axis=-1)
actual_test_y = y_val[24:test_slice]

coverage_80 = empirical_coverage(actual_test_y, q_matrix[:, 0], q_matrix[:, 2])
print(f"\nEmpirical 80% Interval Coverage: {coverage_80 * 100:.2f}% (Target: 80.0%)")

# Plot Prediction Intervals
time_steps = np.arange(len(actual_test_y))
plt.figure(figsize=(12, 5))
plt.fill_between(time_steps, q_matrix[:, 0], q_matrix[:, 2], color='#93c5fd', alpha=0.5, label='80% Prediction Interval [q10, q90]')
plt.plot(time_steps, q_matrix[:, 1], color='#1d4ed8', linewidth=1.8, label='Median Forecast (q50)')
plt.plot(time_steps, actual_test_y, 'o-', color='#111827', markersize=3, linewidth=1.2, label='Actual PM2.5 Observation')
plt.title('Probabilistic Forecasting: 80% Uncertainty Intervals on 1-Week Horizon', fontsize=12, fontweight='bold')
plt.xlabel('Hour Step')
plt.ylabel('PM2.5 Concentration (µg/m³)')
plt.grid(True, linestyle=':', alpha=0.5)
plt.legend(loc='upper right')
plt.tight_layout()
plt.show()


## 8. Comparative Holdout Evaluation: Deep Learning vs. Baselines

We evaluate all 7 models across the complete, identical **8,531 hours in the 2014 holdout period**:
- **Persistence baseline**
- **Daily seasonal naive**
- **Ridge regression** (linear reference)
- **XGBoost** (classical champion)
- **LSTM** (unidirectional neural model)
- **Bi-LSTM** (bidirectional neural model)
- **Calibrated Hybrid** (XGBoost + deep residual correction)

In [ ]:
RUN = ROOT / 'artifacts/phase2_research'
if (RUN / 'metrics.csv').exists():
    metrics_df = pd.read_csv(RUN / 'metrics.csv')
    holdout_metrics = metrics_df[metrics_df['phase'] == 'holdout'].sort_values('rmse').reset_index(drop=True)
    print("2014 Holdout Evaluation Summary (Sorted by RMSE):")
    display(holdout_metrics[['model', 'n', 'mae', 'rmse', 'r2', 'bias']])

    # Bar chart comparison
    plt.figure(figsize=(10, 4.5))
    models = holdout_metrics['model'].str.upper()
    rmse_vals = holdout_metrics['rmse']
    mae_vals = holdout_metrics['mae']
    x = np.arange(len(models))
    width = 0.35

    plt.bar(x - width/2, rmse_vals, width, label='RMSE (µg/m³)', color='#1e3a8a')
    plt.bar(x + width/2, mae_vals, width, label='MAE (µg/m³)', color='#60a5fa')
    plt.axhline(rmse_vals.iloc[0], color='#ef4444', linestyle='--', alpha=0.7, label=f'Best RMSE ({rmse_vals.iloc[0]:.2f})')
    plt.title('Holdout Comparison: Classical vs. Deep Learning Models (2014)', fontsize=12, fontweight='bold')
    plt.xticks(x, models, rotation=25, ha='right')
    plt.ylabel('Error (µg/m³)')
    plt.grid(axis='y', linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 9. Pollution Regime Breakdown & Extreme Smog Events

Error behavior differs markedly across pollution regimes. Models that excel during low/moderate pollution often struggle during extreme inversion spikes ($\ge$ 90th percentile, 218 µg/m³).

In [ ]:
if (RUN / 'errors_by_pollution_regime.csv').exists():
    regimes_df = pd.read_csv(RUN / 'errors_by_pollution_regime.csv')
    above_p90 = regimes_df[regimes_df['pollution_regime'] == 'above_training_p90'].sort_values('rmse')
    print("High Pollution Regime Performance (PM2.5 >= 218 µg/m³):")
    display(above_p90[['model', 'n', 'mae', 'rmse', 'r2', 'bias']])

    # Visualization
    plt.figure(figsize=(9, 4))
    plt.barh(above_p90['model'].str.upper(), above_p90['rmse'], color='#b91c1c')
    plt.title('RMSE during Hazardous Smog Hours (>= 90th Percentile: 218 µg/m³)', fontsize=12, fontweight='bold')
    plt.xlabel('RMSE (µg/m³)')
    plt.grid(axis='x', linestyle=':', alpha=0.6)
    plt.tight_layout()
    plt.show()


## 10. Ablation Studies & Seed Stability Analysis

We review the ablation results testing:
1. **Attention vs. No-Attention**
2. **Unidirectional vs. Bidirectional LSTM**
3. **Random seed stability across seeds 42, 137, and 2026**

In [ ]:
for fname in ('ablation_metrics.csv', 'seed_stability.csv'):
    if (RUN / fname).exists():
        print(f"\n--- {fname} ---")
        display(pd.read_csv(RUN / fname))


## 11. Saved Research Artifacts & Report Figures

Inspect the final published high-resolution scientific evaluation figures generated by the project pipeline.

In [ ]:
FIGURES = ROOT / 'docs/results/phase2_research'
for fig_name in ('model_comparison.png', 'attention_ablation.png', 'pollution_regimes.png'):
    fig_path = FIGURES / fig_name
    if fig_path.exists() and NotebookImage is not None:
        print(f"Displaying: {fig_name}")
        display(NotebookImage(filename=str(fig_path)))


---
**Citation & Attribution:**
- Dataset: Chen, S. (2015), *Beijing PM2.5*, UCI Machine Learning Repository, DOI [10.24432/C5JS49](https://doi.org/10.24432/C5JS49), CC BY 4.0.
- Codebase & Full Documentation: [Aircast Project Repository](https://github.com/pratham-malhotr/air_prediction)